In [3]:
from pyspark.sql import SparkSession


In [4]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

In [5]:
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [6]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [8]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [9]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [13]:
df.describe().show()

+-------+-----------------+-----------+-------+-------+-------+
|summary|           amount|   category|  store|  tx_id|user_id|
+-------+-----------------+-----------+-------+-------+-------+
|  count|            10000|      10000|  10000|  10000|  10000|
|   mean|401.1544750000007|       NULL|   NULL|   NULL|   NULL|
| stddev|635.5781805674461|       NULL|   NULL|   NULL|   NULL|
|    min|              5.0|elektronika| Gdańsk|TX00001|    u01|
|    max|           9999.0|    żywność|Wrocław|TX10000|    u50|
+-------+-----------------+-----------+-------+-------+-------+



In [17]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round, min as _min, max as _max

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)

#2.2
cat_summary = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("minumim_PLN"),
        _round(_max("amount"), 2).alias("maksimum_PLN"),
    )
    .orderBy("category").show()
)


+-----------+---------+----------+-----------+------------+
|   category|liczba_tx|  suma_PLN|minumim_PLN|maksimum_PLN|
+-----------+---------+----------+-----------+------------+
|elektronika|     2542|1520770.69|        9.0|      9999.0|
|    książki|     2574| 851382.08|        5.0|     9107.25|
|     odzież|     2453| 849877.55|        5.0|     9696.63|
|    żywność|     2431| 789514.43|        5.0|     6916.92|
+-----------+---------+----------+-----------+------------+



In [15]:
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [19]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
#3.2
half_hourly = (
    df.groupBy(window("timestamp", "30 minutes"), "store")    # okno 30-minutowe
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window").show()
)

+--------------------+--------+---------+---------+
|              window|   store|liczba_tx| suma_PLN|
+--------------------+--------+---------+---------+
|{2026-04-12 08:00...| Wrocław|      296|111540.59|
|{2026-04-12 08:00...|  Gdańsk|      252| 93391.22|
|{2026-04-12 08:00...|  Kraków|      289|117786.42|
|{2026-04-12 08:00...|Warszawa|      275| 88441.58|
|{2026-04-12 08:30...|  Gdańsk|      514|209187.85|
|{2026-04-12 08:30...| Wrocław|      502|215587.17|
|{2026-04-12 08:30...|  Kraków|      532|223541.41|
|{2026-04-12 08:30...|Warszawa|      490|182435.06|
|{2026-04-12 09:00...|  Kraków|      590|224358.03|
|{2026-04-12 09:00...|Warszawa|      584|214573.66|
|{2026-04-12 09:00...| Wrocław|      612|229985.47|
|{2026-04-12 09:00...|  Gdańsk|      619|253364.95|
|{2026-04-12 09:30...|  Gdańsk|      555|234914.63|
|{2026-04-12 09:30...|  Kraków|      579|258951.83|
|{2026-04-12 09:30...|Warszawa|      533|237064.34|
|{2026-04-12 09:30...| Wrocław|      589| 243017.3|
|{2026-04-12

In [31]:
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [28]:
from pyspark.sql.functions import desc

# 3.3
hourly_krk = (
    df.filter(df.store == "Kraków").groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy(desc("suma_PLN")).show()
)


+--------------------+---------+---------+
|              window|liczba_tx| suma_PLN|
+--------------------+---------+---------+
|{2026-04-12 09:00...|     1169|483309.86|
|{2026-04-12 08:00...|      821|341327.83|
|{2026-04-12 10:00...|      532|201259.26|
+--------------------+---------+---------+



In [26]:
GDlow = (
    df.filter(df.store == "Gdańsk").groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx_gd"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("srednia_PLN")
)

In [27]:
GDlow.show(truncate=False)

+------------------------------------------+------------+-----------+
|window                                    |liczba_tx_gd|srednia_PLN|
+------------------------------------------+------------+-----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|766         |395.01     |
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|558         |412.92     |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|1174        |415.91     |
+------------------------------------------+------------+-----------+



In [1]:
%%file streamrate.py
## uruchom przez spark-submit streamrate.py

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)


query = (df.writeStream 
    .format("console") 
    .outputMode("append") 
    .option("truncate", False) 
    .start()
) 

query.awaitTermination()

Writing streamrate.py


In [29]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [30]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# Sliding ma więcej bo zaczyna się wcześniej i kończy później oraz ma okna między połowami godzin

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [32]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: Pierwsze grupuje po sklepie, drugie po oknie i w ramach okna po sklepie

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 - 9:00-10:00, 9:30-10:30